# A1 → A2 → A3: M1 DiffMul gốc + M2 B5B6 (focal + augmentation + SupCon)

- M1: CodeBERT DiffMul gốc (best checkpoint).
- M2: GraphCodeBERT DiffMul B5/B6 (best checkpoint).
- A1: calibration isotonic + ngưỡng 5-fold OOF trên validation.
- A2: SWA best+last (bỏ qua nếu thiếu last).
- A3: ensemble M1 + M2, trọng số quét trên OOF validation.

Checkpoint lấy từ 2 dataset Kaggle; parquet val/test từ dataset hoặc tải từ HF.


In [ ]:
# 1. SETUP + STREAM LOG
import base64, io, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

WORK = Path('/kaggle/working')
CLEAN = WORK / 'clean_postprocess'
CLEAN.mkdir(parents=True, exist_ok=True)

def sh(cmd):
    print(f'\n$ {cmd}', flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    log_file = WORK / 'live_progress.log'
    with log_file.open('a', encoding='utf-8') as lf:
        for line in iter(proc.stdout.readline, ''):
            print(line, end='', flush=True)
            lf.write(line)
            lf.flush()
    proc.stdout.close()
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'Lenh that bai (rc={rc}): {cmd}')

print('Python:', sys.version.split()[0], flush=True)


In [ ]:
# 2. GIAI NEN MA NGUON
payload = '...'
repo_root = WORK / 'project'
repo_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(payload)), mode='r:gz') as tar:
    tar.extractall(repo_root)
os.chdir(repo_root)
print('Repo:', os.getcwd(), flush=True)
sh('ls scripts/postprocess_diffmul.py src/postprocess/pipeline.py')

In [ ]:
# 3. GPU + DEPS
import torch
print('CUDA:', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available():
    raise SystemError('Kernel chua bat GPU!')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
sh('pip install -q -U transformers datasets scikit-learn scipy pyarrow')

In [ ]:
# 4. LINK PARQUET VAL/TEST VA CHECKPOINT TUNG DATASET KAGGLE
input_path = Path('/kaggle/input')
print('Inputs:', [p.name for p in input_path.iterdir()], flush=True)

data_raw = repo_root / 'data/raw/bigclonebench'
data_raw.mkdir(parents=True, exist_ok=True)
for name in ('bcb_validation.parquet', 'bcb_test.parquet'):
    cands = list(input_path.rglob(name))
    dest = data_raw / name
    if cands and not dest.exists():
        sh(f'cp "{cands[0]}" "{dest}"')
    print(name, 'exists=', dest.exists(), flush=True)
if any(not (data_raw / n).exists() for n in ('bcb_validation.parquet', 'bcb_test.parquet')):
    print('Tai lai splits tu Hugging Face...', flush=True)
    sh('python scripts/download_bcb_subsets.py --splits-custom --train-samples 30000 --val-samples 3000 --test-samples 5000')
assert (data_raw / 'bcb_validation.parquet').exists(), 'Thieu bcb_validation.parquet!'
assert (data_raw / 'bcb_test.parquet').exists(), 'Thieu bcb_test.parquet!'

# Checkpoint tu dataset (tim theo ten file)
ckpt_map = {
    'm1_best_checkpoint.pt': 'results/exp_m1_diffmul/best_checkpoint.pt',
    'm2_b5b6_best_checkpoint.pt': 'results/exp_m2_b5b6/best_checkpoint.pt',
}
found = {}
for f in input_path.rglob('*.pt'):
    if f.name in ckpt_map and f.name not in found:
        found[f.name] = f
        print('Dataset ckpt:', f.name, f.parent.parent.name, flush=True)

for name, dest in ckpt_map.items():
    dest_path = repo_root / dest
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    if name in found and not dest_path.exists():
        shutil.copy2(found[name], dest_path)
    p = Path(dest)
    print(dest, 'exists=', p.exists(), f', {p.stat().st_size/1e6:.1f} MB' if p.exists() else '', flush=True)
    assert p.exists(), f'Thieu checkpoint {dest}'

In [ ]:
# 5. CHAY A1 -> A2 -> A3
print('=' * 70)
print('>>> BAT DAU POSTPROCESS M1 (goc) + M2 (B5B6) <<<')
print('=' * 70)
cmd = (
    'python scripts/postprocess_diffmul.py '
    '--val-file data/raw/bigclonebench/bcb_validation.parquet '
    '--test-file data/raw/bigclonebench/bcb_test.parquet '
    '--out-dir results/exp_diffmul_postprocess_m1_m2b5b6 '
    '--batch-size 32 --precision fp16 --n-folds 5 --seed 42 '
    '--m1-best results/exp_m1_diffmul/best_checkpoint.pt '
    '--m2-best results/exp_m2_b5b6/best_checkpoint.pt'
)
t0 = time.perf_counter()
sh(cmd)
print(f'>>> HOAN TAT TRONG {(time.perf_counter()-t0)/60:.1f} PHUT!', flush=True)
print((repo_root / 'results/exp_diffmul_postprocess_m1_m2b5b6/postprocess_report.md').read_text(encoding='utf-8'), flush=True)

In [ ]:
# 6. THU THAP KET QUA + AUTO-CLEANUP
WORK = Path('/kaggle/working')
CLEAN = WORK / 'clean_postprocess'
CLEAN.mkdir(parents=True, exist_ok=True)
out_dir = repo_root / 'results' / 'exp_diffmul_postprocess_m1_m2b5b6'

for p in sorted(out_dir.rglob('*')):
    if p.is_file() and p.suffix in ('.json', '.md', '.npy', '.log'):
        name = 'postprocess_' + p.name
        shutil.copy2(p, CLEAN / name)
        shutil.copy2(p, WORK / name)
        print('  [SAVED]', name, flush=True)

log_file = WORK / 'live_progress.log'
if log_file.exists():
    shutil.copy2(log_file, CLEAN / 'postprocess_live_progress.log')
    shutil.copy2(log_file, WORK / 'postprocess_live_progress.log')

with tarfile.open(WORK / 'diffmul_postprocess_m1_m2b5b6_bundle.tar.gz', 'w:gz') as tar:
    for f in CLEAN.glob('*'):
        tar.add(f, arcname=f.name)
print('>>> Bundle:', WORK / 'diffmul_postprocess_m1_m2b5b6_bundle.tar.gz', flush=True)

os.chdir(WORK)
for junk in ('project', 'data', 'artifacts', 'clean_postprocess'):
    shutil.rmtree(WORK / junk, ignore_errors=True)
print('DANH SACH OUTPUT:', flush=True)
for p in sorted(WORK.glob('postprocess_*')):
    print(f'  {p.name} ({p.stat().st_size/1e6:.2f} MB)', flush=True)
print('>>> XONG 100%!', flush=True)